In [ ]:
# --------------------------------------------------
# Project Root
# --------------------------------------------------

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\IITG\Projects\audio_factor_disentanglement_v2


In [6]:
# --------------------------------------------------
# Imports
# --------------------------------------------------

import pandas as pd
from tqdm import tqdm

from src.utils.config_loader import load_yaml
from src.utils.path_manager import PathManager
from src.utils.metadata_manager import MetadataManager

from src.preprocessing.audio_standardizer import (
    AudioStandardizer
)

from src.utils.file_utils import (
    replace_root_and_extension
)


In [4]:
# --------------------------------------------------
# Load Config
# --------------------------------------------------

cfg = load_yaml(
    PROJECT_ROOT
    / "configs"
    / "data_config.yaml"
)

cfg

{'project_name': 'audio_factor_disentanglement_v2',
 'audio_extensions': ['.wav', '.ogg', '.m4a'],
 'dataset': {'raw_dir': 'data/raw',
  'processed_dir': 'data/processed',
  'metadata_dir': 'data/metadata',
  'fragment_dir': 'data/fragments',
  'feature_dir': 'data/features'},
 'outputs': {'figures_dir': 'outputs/figures',
  'checkpoint_dir': 'outputs/checkpoints',
  'swap_dir': 'outputs/swaps'},
 'audio': {'target_sr': 16000, 'mono': True, 'normalize_peak': True},
 'fragmentation': {'sample_rate': 16000,
  'max_fragment_ms': 500,
  'min_fragment_ms': 80,
  'padding_samples': 8192,
  'vad_top_db': 25,
  'merge_gap_ms': 50}}

In [7]:
# --------------------------------------------------
# Setup Paths
# --------------------------------------------------

pm = PathManager(
    project_root=PROJECT_ROOT,
    config=cfg
)

RAW_DIR = pm.get_path(
    cfg["dataset"]["raw_dir"]
)

PROCESSED_DIR = pm.get_path(
    cfg["dataset"]["processed_dir"]
)

METADATA_DIR = pm.get_path(
    cfg["dataset"]["metadata_dir"]
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("RAW_DIR:")
print(RAW_DIR)

print("\nPROCESSED_DIR:")
print(PROCESSED_DIR)

print("\nMETADATA_DIR:")
print(METADATA_DIR)

RAW_DIR:
d:\IITG\Projects\audio_factor_disentanglement_v2\data\raw

PROCESSED_DIR:
d:\IITG\Projects\audio_factor_disentanglement_v2\data\processed

METADATA_DIR:
d:\IITG\Projects\audio_factor_disentanglement_v2\data\metadata


In [5]:
# --------------------------------------------------
# Load Phase-0 Inventory
# --------------------------------------------------

inventory = MetadataManager.load_csv(
    METADATA_DIR
    / "audio_inventory.csv"
)

print(
    f"Inventory Size: {len(inventory)}"
)

inventory.head()

Inventory Size: 34


,filepath,filename,split,speaker,condition,extension,sample_rate,channels,duration_sec,num_samples,rms,peak
0,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_01.m4a,train,s1,clean,.m4a,48000,2,2.090667,100352,0.043963,0.314957
1,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_02.m4a,train,s1,clean,.m4a,48000,2,2.133333,102400,0.044887,0.301422
2,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_03.m4a,train,s1,clean,.m4a,48000,2,1.920000,92160,0.052497,0.321243
3,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_04.m4a,train,s1,clean,.m4a,48000,2,1.962667,94208,0.047852,0.279785
4,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_05.m4a,train,s1,clean,.m4a,48000,2,2.133333,102400,0.050404,0.328415


In [6]:
# --------------------------------------------------
# Standardizer
# --------------------------------------------------

standardizer = AudioStandardizer(

    target_sr=
        cfg["audio"]["target_sr"],

    mono=
        cfg["audio"]["mono"],

    peak_normalize=
        cfg["audio"]["normalize_peak"]
)

In [7]:
# --------------------------------------------------
# Standardize Dataset
# --------------------------------------------------

records = []

for _, row in tqdm(
    inventory.iterrows(),
    total=len(inventory)
):

    source_file = Path(
        row["filepath"]
    )

    target_file = replace_root_and_extension(

        source_path=source_file,

        source_root=RAW_DIR,

        target_root=PROCESSED_DIR,

        new_extension=".wav"
    )

    result = standardizer.process_file(

        input_path=source_file,

        output_path=target_file
    )

    result["speaker"] = row["speaker"]

    result["condition"] = row["condition"]

    result["split"] = row["split"]

    records.append(result)

  0%|          | 0/34 [00:00<?, ?it/s]c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\IITG\Projects\audio_factor_disentanglement_v2\src\utils\audio_loader.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  3%|▎         | 1/34 [00:06<03:44,  6.80s/it]d:\IITG\Projects\audio_factor_disentanglement_v2\src\utils\audio_loader.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\li

In [8]:
# --------------------------------------------------
# Build Metadata
# --------------------------------------------------

standardized_df = pd.DataFrame(
    records
)

print(
    f"Standardized Files: "
    f"{len(standardized_df)}"
)

standardized_df.head()

Standardized Files: 34


,original_path,processed_path,original_sr,processed_sr,original_channels,processed_channels,duration_sec,rms_before,rms_after,peak_before,peak_after,speaker,condition,split
0,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...,48000,16000,2,1,2.090688,0.043961,0.141931,0.309737,1.0,s1,clean,train
1,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...,48000,16000,2,1,2.133375,0.044886,0.148792,0.301668,1.0,s1,clean,train
2,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...,48000,16000,2,1,1.920000,0.052495,0.162351,0.323342,1.0,s1,clean,train
3,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...,48000,16000,2,1,1.962687,0.047763,0.170687,0.279826,1.0,s1,clean,train
4,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...,48000,16000,2,1,2.133375,0.050399,0.155940,0.323192,1.0,s1,clean,train


In [9]:
# --------------------------------------------------
# Save Metadata
# --------------------------------------------------

csv_path = (
    METADATA_DIR
    / "audio_inventory_standardized.csv"
)

MetadataManager.save_csv(
    standardized_df,
    csv_path
)

print(csv_path)

d:\IITG\Projects\audio_factor_disentanglement_v2\data\metadata\audio_inventory_standardized.csv


In [10]:
# --------------------------------------------------
# Verify Sample Rates
# --------------------------------------------------

standardized_df[
    "processed_sr"
].value_counts()

processed_sr
16000    34
Name: count, dtype: int64

In [11]:
# --------------------------------------------------
# Verify Channels
# --------------------------------------------------

standardized_df[
    "processed_channels"
].value_counts()

processed_channels
1    34
Name: count, dtype: int64

In [12]:
# --------------------------------------------------
# Verify Processed Files
# --------------------------------------------------

processed_files = list(
    PROCESSED_DIR.rglob("*.wav")
)

print(
    f"Processed WAV Files: "
    f"{len(processed_files)}"
)

Processed WAV Files: 34


In [13]:
# --------------------------------------------------
# Statistics
# --------------------------------------------------

display(

    standardized_df[
        [
            "duration_sec",
            "rms_before",
            "rms_after",
            "peak_before",
            "peak_after"
        ]
    ].describe()
)

,duration_sec,rms_before,rms_after,peak_before,peak_after
count,34.000000,34.000000,34.000000,34.000000,34.0
mean,2.336798,0.089932,0.156966,0.617721,1.0
std,0.359620,0.049342,0.031965,0.366358,0.0
min,1.920000,0.032928,0.107411,0.177693,1.0
25%,2.077812,0.042173,0.133373,0.283501,1.0
50%,2.238250,0.082991,0.153232,0.629761,1.0
75%,2.477031,0.137877,0.165620,0.989975,1.0
max,3.337813,0.165740,0.231794,1.000000,1.0


In [14]:
# --------------------------------------------------
# Train/Test Distribution
# --------------------------------------------------

display(

    standardized_df
    .groupby(
        [
            "split",
            "speaker",
            "condition"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

,split,speaker,condition,count
0,test,s1,clean,1
1,test,s2,noisy,1
2,train,s1,clean,8
3,train,s1,noisy,8
4,train,s2,clean,8
5,train,s2,noisy,8


In [15]:
# --------------------------------------------------
# Inspect Example Paths
# --------------------------------------------------

standardized_df[
    [
        "original_path",
        "processed_path"
    ]
].head(10)

,original_path,processed_path
0,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
1,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
2,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
3,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
4,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
5,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
6,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
7,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
8,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
9,d:\IITG\Projects\audio_factor_disentanglement_...,d:\IITG\Projects\audio_factor_disentanglement_...
